<a href="https://colab.research.google.com/github/vidhikajain10/AI_TASK/blob/main/CNN_Fashion_MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Check GPU
import torch
print("GPU available:", torch.cuda.is_available())


GPU available: True


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [11]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_data = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)


In [12]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(1600, 128)

        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)

        x = F.relu(self.conv2(x))
        x = self.pool(x)

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 5

for epoch in range(epochs):
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss:.4f}")


Epoch 1/5, Loss: 449.8888
Epoch 2/5, Loss: 301.3169
Epoch 3/5, Loss: 253.2427
Epoch 4/5, Loss: 223.7621
Epoch 5/5, Loss: 198.8273


In [13]:
correct = 0
total = 0

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")


Test Accuracy: 91.04%


In [14]:
"""
Conclusion:
- Built a CNN-based image classifier using PyTorch
- Trained on Fashion-MNIST dataset
- Achieved ~90% accuracy
- Implemented full pipeline: data → model → training → evaluation
"""


'\nConclusion:\n- Built a CNN-based image classifier using PyTorch\n- Trained on Fashion-MNIST dataset\n- Achieved ~90% accuracy\n- Implemented full pipeline: data → model → training → evaluation\n'

In [15]:
!pip install gradio


In [16]:
class_names = [
    "T-shirt/Top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle Boot"
]


In [17]:
from PIL import Image
import numpy as np

def preprocess_image(img):
    # Convert to grayscale
    img = img.convert("L")

    # Resize to 28x28
    img = img.resize((28, 28))

    # Convert to tensor
    img = transforms.ToTensor()(img)

    # Normalize (same as training)
    img = transforms.Normalize((0.5,), (0.5,))(img)

    # Add batch dimension
    img = img.unsqueeze(0)

    return img


In [18]:
import torch.nn.functional as F

def predict_image(img):
    model.eval()
    img = preprocess_image(img).to(device)

    with torch.no_grad():
        outputs = model(img)
        probs = F.softmax(outputs, dim=1)[0]

    # Convert to dictionary for Gradio
    confidence_dict = {
        class_names[i]: float(probs[i])
        for i in range(len(class_names))
    }

    return confidence_dict


In [19]:
import gradio as gr

interface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(num_top_classes=3),
    title="CNN Fashion Image Classifier",
    description="Upload an image and view prediction confidence."
)



interface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8feb406c4292088177.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
